In [ ]:
import glob, os, subprocess, sys
hits = glob.glob("/kaggle/input/**/qwen38-wheelhouse-py312", recursive=True)
WH = hits[0] if hits else None
print("WHEELHOUSE:", WH, flush=True)
print("INPUT_ROOT:", sorted(glob.glob("/kaggle/input/*")), flush=True)
if WH:
    print(sorted(os.listdir(WH)), flush=True)
r0 = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True, text=True)
print((r0.stdout or "")[-500:], flush=True)
if not WH:
    raise SystemExit("NO_WHEELHOUSE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links=%s" % WH, "transformers==5.8.0", "accelerate", "peft"], capture_output=True, text=True)
print((r.stdout or "")[-1500:], flush=True)
print((r.stderr or "")[-500:], flush=True)
if r.returncode != 0:
    raise SystemExit("WHEEL_INSTALL_FAILED %s" % r.returncode)
import transformers
print("TF_NOW:", transformers.__version__, hasattr(transformers, "Qwen3_5ForCausalLM"), flush=True)


In [ ]:
import glob, os, sys, subprocess
print("== 007 G1 grid SFT ==", flush=True)

def find_code_dir():
    bases = [os.getcwd(), "/kaggle/working", "/kaggle/src", "/kaggle/input"]
    found = []
    for base in bases:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            dirnames[:] = [name for name in dirnames if name not in (".git", "__pycache__")]
            if "train_grid_sft.py" in filenames and "arc_episode_data.py" in filenames and "episodes.jsonl" in filenames:
                found.append(dirpath)
            depth = dirpath.count(os.sep) - base.count(os.sep)
            if depth >= 4:
                dirnames.clear()
    return found

code_dirs = find_code_dir()
print("CODE_DIRS:", code_dirs, flush=True)
if not code_dirs:
    raise SystemExit("CODE_NOT_FOUND")
code_dir = code_dirs[0]
ch = glob.glob("/kaggle/input/competitions/**/arc-agi_training_challenges.json", recursive=True)
so = glob.glob("/kaggle/input/competitions/**/arc-agi_training_solutions.json", recursive=True)
print("CHALLENGES:", ch, flush=True)
print("SOLUTIONS:", so, flush=True)
if not ch or not so:
    raise SystemExit("COMPETITION_JSON_NOT_FOUND")
out_dir = "/kaggle/working/007-g1"
os.makedirs(out_dir, exist_ok=True)
cmd = [
    sys.executable,
    os.path.join(code_dir, "train_grid_sft.py"),
    "--episodes", os.path.join(code_dir, "episodes.jsonl"),
    "--out-dir", out_dir,
    "--challenges", ch[0],
    "--solutions", so[0],
    "--optimizer-updates", "200",
    "--grad-accum", "4",
    "--microbatch", "1",
    "--max-len", "1024",
    "--overflow-len", "2048",
    "--save-every", "25",
    "--lr", "2e-5",
    "--weight-decay", "0.01",
    "--warmup-updates", "10",
    "--clip-norm", "1.0",
    "--seed", "20260930",
    "--zero-step-max-new", "960",
]
print("CMD:", " ".join(cmd), flush=True)
proc = subprocess.run(cmd)
print("TRAIN_EXIT", proc.returncode, flush=True)
summary = os.path.join(out_dir, "SUMMARY.json")
print("SUMMARY_EXISTS", os.path.isfile(summary), flush=True)
if os.path.isfile(summary):
    print(open(summary, encoding="utf-8").read()[:4000], flush=True)
if proc.returncode not in (0, 2):
    raise SystemExit(proc.returncode)
